# Early-Warning System for Risk-Off Regimes + Allocation Routing
## Graduate School of Management — Politecnico di Milano

**Objective:** Build an anomaly-detection-based Early-Warning System (EWS) that flags risk-off weeks
on weekly Bloomberg data, routes the signal into regime-specific allocations (LEVERED_EQUITY / CASH_USD / GOLD / MBS),
and backtests the strategy against 60/40 and buy-and-hold benchmarks, including a COVID stress test.

**Methodology:** Unsupervised novelty detection (MVG, One-Class SVM, Autoencoder, Isolation Forest)
trained on normal (Y=0) weeks, combined in an ensemble, validated with purged expanding walk-forward CV + embargo,
evaluated on a sealed 2019-2021 test holdout containing COVID.


## 1. Environment Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')
import os, sys
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Ensure src/ is importable
sys.path.insert(0, os.path.abspath('..'))
if not os.path.exists('src'):
    os.chdir('..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
print("Setup complete.")


Setup complete.


## 2. Data Loading & EDA

The dataset contains 1,111 weekly observations from Bloomberg (2000-01-11 to 2021-04-20),
with 43 features spanning equities (MSCI indices), FX, commodities, bond indices, yields, and rates.
Target Y = 1 indicates risk-off weeks (~21.3% prevalence).


In [2]:
from src.data_loader import load_dataset, TARGET_COL
df_raw = load_dataset()


Dataset: 1111 rows x 43 cols (2000-01-11 → 2021-04-20)
Target prevalence: Y=1 237/1111 (21.3%)
          first_valid last_valid  n_obs  n_missing  pct_missing
feature                                                        
BDIY       2000-01-11 2021-04-20   1111          0          0.0
CRY        2000-01-11 2021-04-20   1111          0          0.0
Cl1        2000-01-11 2021-04-20   1111          0          0.0
DXY        2000-01-11 2021-04-20   1111          0          0.0
ECSURPUS   2000-01-11 2021-04-20   1111          0          0.0
EMUSTRUU   2000-01-11 2021-04-20   1111          0          0.0
EONIA      2000-01-11 2021-04-20   1111          0          0.0
GBP        2000-01-11 2021-04-20   1111          0          0.0
GT10       2000-01-11 2021-04-20   1111          0          0.0
GTDEM10Y   2000-01-11 2021-04-20   1111          0          0.0
GTDEM2Y    2000-01-11 2021-04-20   1111          0          0.0
GTDEM30Y   2000-01-11 2021-04-20   1111          0          0.0
GTGBP20Y 

In [3]:
# Y distribution by year
yearly = df_raw.groupby(df_raw.index.year)[TARGET_COL].agg(['sum', 'count'])
yearly.columns = ['n_riskoff', 'n_total']
yearly['pct_riskoff'] = (100 * yearly['n_riskoff'] / yearly['n_total']).round(1)
print(yearly)


      n_riskoff  n_total  pct_riskoff
Date                                 
2000         19       51         37.3
2001         24       52         46.2
2002         27       53         50.9
2003          6       52         11.5
2004          0       52          0.0
2005          0       52          0.0
2006          2       52          3.8
2007          6       52         11.5
2008         27       53         50.9
2009         20       52         38.5
2010         14       52         26.9
2011         24       52         46.2
2012         27       52         51.9
2013          0       53          0.0
2014          2       52          3.8
2015          6       52         11.5
2016          4       52          7.7
2017          0       52          0.0
2018          6       52         11.5
2019          0       53          0.0
2020         23       52         44.2
2021          0       16          0.0


## 3. Stationarity Transforms

- **Log-returns** for price-like series (equities, FX, commodities, bond total-return indices)
- **First differences** for yields and rates
- **Level** for VIX, ECSURPUS, and target Y

All 42 features pass the ADF test at 5% significance.


In [4]:
from src.features import make_stationary, adf_table, TRANSFORM_MAP
df_stat = make_stationary(df_raw)
adf = adf_table(df_stat)
n_stat = adf['stationary'].sum()
print(f"Stationary features: {n_stat}/{len(adf)}")
print(adf[['adf_stat', 'p_value', 'transform', 'stationary']].to_string())


Stationary features: 42/42
            adf_stat       p_value   transform  stationary
feature                                                   
BDIY       -8.474755  1.445470e-13  log_return        True
CRY       -10.024276  1.647770e-17  log_return        True
Cl1       -13.715296  1.216108e-25  log_return        True
DXY       -33.497252  0.000000e+00  log_return        True
ECSURPUS   -6.480751  1.296887e-08       level        True
EMUSTRUU  -16.529793  2.028485e-29  log_return        True
EONIA      -5.544448  1.673126e-06        diff        True
GBP       -18.831036  0.000000e+00  log_return        True
GT10      -33.795264  0.000000e+00        diff        True
GTDEM10Y  -25.265759  0.000000e+00        diff        True
GTDEM2Y    -7.622819  2.106366e-11        diff        True
GTDEM30Y  -26.777823  0.000000e+00        diff        True
GTGBP20Y  -37.637565  0.000000e+00        diff        True
GTGBP2Y    -9.073669  4.235464e-15        diff        True
GTGBP30Y  -38.280705  0.00000

## 4. Cross-Asset Spreads & Feature Engineering

20 spread features constructed: term spreads, credit spreads (log price-index ratios),
equity-bond rotation, gold-oil ratio, VRP, JPY strength. Each with level + 4-week change.


In [5]:
from src.features import build_spreads
spreads = build_spreads(df_raw)
print(f"Spreads: {spreads.shape}")
print(spreads.describe().T[['mean', 'std', 'min', 'max']])


Spreads: (1107, 20)
                           mean        std        min         max
us_term_10y_3m         1.693516   1.162708  -0.988000    3.805100
us_term_10y_3m_chg4w   0.001671   0.272180  -1.137400    1.191000
us_term_10y_2y         1.280264   0.902131  -0.485000    2.890200
us_term_10y_2y_chg4w   0.004620   0.164621  -0.793600    0.682000
de_term_10y_2y         1.016481   0.582474  -0.138000    2.358000
de_term_10y_2y_chg4w  -0.002962   0.141016  -0.420000    0.704000
it_de_10y              1.235997   1.083873   0.044000    5.337000
it_de_10y_chg4w        0.002782   0.262666  -1.599000    1.720000
us_de_10y              0.817224   0.853465  -0.894000    2.796000
us_de_10y_chg4w        0.002845   0.156799  -0.763000    0.606000
hy_spread              0.358003   0.225163  -0.029378    0.939723
hy_spread_chg4w       -0.001784   0.028829  -0.147929    0.232420
hy_ig_spread           0.574025   0.146355   0.346340    0.940268
hy_ig_spread_chg4w    -0.000838   0.022546  -0.116859   

## 5. Collinearity Cleanup & Routing Triggers

7 redundant features removed (|corr| > 0.9). 14 routing triggers constructed across
3 domains: USD (5), Oro (5), MBS (4+1 shared with USD).


In [6]:
from src.features import (remove_collinear_features, build_routing_triggers,
                          COLLINEAR_DROP, FEATURES_CLEAN_PATH, SPREADS_CLEAN_PATH)

df_stat_clean = remove_collinear_features(df_stat)
spreads_clean = remove_collinear_features(spreads)
print(f"Clean features: {df_stat_clean.shape}")
print(f"Clean spreads: {spreads_clean.shape}")

triggers = build_routing_triggers(df_raw, df_stat, spreads)
print(f"Routing triggers: {triggers.shape}")
print(f"Dropped collinear: {COLLINEAR_DROP}")


Clean features: (1110, 40)
Clean spreads: (1107, 16)
Routing triggers: (1060, 14)
Dropped collinear: ['hy_spread', 'hy_spread_chg4w', 'us_term_10y_3m', 'us_term_10y_3m_chg4w', 'GTGBP20Y', 'GTDEM30Y', 'USGG30YR']


## 6. Walk-Forward Cross-Validation

5-fold expanding walk-forward with 4-week embargo. Development set ends 2018-12-31.
Test holdout: 2019-2021 (contains COVID).

| Fold | Crisis | Val Y=1% |
|------|--------|----------|
| 1 | GFC 2008 | 34.6% |
| 2 | Euro 2011 | 42.1% |
| 3 | Taper 2013 | 2.0% |
| 4 | China-Oil 2015-16 | 10.0% |
| 5 | Q4 2018 selloff | 6.1% |


In [7]:
from src.splits import walkforward_split
wf = walkforward_split(embargo_weeks=4, save=False)


Model splits:
  Fold 1 (GFC 2008): train=360, val=152, Y=1 53 (34.9%)
  Fold 2 (Euro 2011): train=517, val=151, Y=1 63 (41.7%)
  Fold 3 (Taper 2013): train=673, val=100, Y=1 2 (2.0%) ⚠️ WARN low prevalence ⚠️ few positives
  Fold 4 (China-Oil 2015-16): train=778, val=99, Y=1 10 (10.1%) ℹ️ NOTE ⚠️ few positives
  Fold 5 (Q4 2018 selloff): train=882, val=99, Y=1 6 (6.1%) ⚠️ WARN low prevalence ⚠️ few positives
  ✓ COVID in test holdout (13 rows)

Routing splits:


## 7. Anomaly Detection Models

Four models trained on **normal weeks only (Y=0)**, with thresholds tuned via walk-forward CV weighted by n_pos:

1. **MVG** (Ledoit-Wolf shrinkage) — squared Mahalanobis distance
2. **One-Class SVM** (RBF kernel) — grid search over nu x gamma
3. **Autoencoder** (56→24→12→6→12→24→56) — reconstruction MSE
4. **Isolation Forest** (200 trees) — grid search over contamination


In [8]:
from src.models import run_all_models
all_models, results, final_scaler, fold_data, scalers = run_all_models(verbose=True)


MVG (Ledoit-Wolf)


  Threshold: 460.34, weighted F1: 0.653
  Test: F1=0.516, AUC_PR=0.770, AUC_ROC=0.886, Prec=1.000, Rec=0.348, F2=0.400
One-Class SVM (grid search)


  Best params: {'nu': 0.05, 'gamma': 0.001}, CV weighted F1: 0.596
  Test: F1=0.615, AUC_PR=0.732, AUC_ROC=0.851, Prec=0.750, Rec=0.522, F2=0.556
Isolation Forest (grid search)


  Best params: {'contamination': 0.1}, CV weighted F1: 0.573


  Tuned threshold: -0.0086, weighted F1: 0.570
  Test: F1=0.583, AUC_PR=0.680, AUC_ROC=0.810, Prec=0.560, Rec=0.609, F2=0.598
Autoencoder


I0000 00:00:1779806663.936155   25863 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779806663.936782   25863 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1779806665.517879   25863 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779806665.518292   25863 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


  Threshold: 3.712852, weighted F1: 0.634
  Test: F1=0.467, AUC_PR=0.743, AUC_ROC=0.863, Prec=1.000, Rec=0.304, F2=0.354

MODEL COMPARISON (Test Holdout)
           F1  Precision    Recall        F2   AUC_ROC    AUC_PR
MVG  0.516129       1.00  0.347826  0.400000  0.885701  0.770444
SVM  0.615385       0.75  0.521739  0.555556  0.850740  0.731559
IF   0.583333       0.56  0.608696  0.598291  0.809951  0.679511
AE   0.466667       1.00  0.304348  0.353535  0.862842  0.743406


## 8. Ensemble Detection

Three ensemble variants:
- **Hard voting** (majority 3/4)
- **Soft voting mean** (percentile-mapped scores)
- **Soft voting median** (robust to outlier model)

Scores are percentile-mapped against the train-normals distribution to make them comparable.


In [9]:
from src.ensemble import run_ensemble
ensemble, ens_results, pred_series, score_series = run_ensemble(
    all_models, results, final_scaler, fold_data, scalers, verbose=True
)


Soft mean threshold: 0.936
Soft median threshold: 0.936



Error correlation analysis:
  Mean pairwise error correlation: 0.649
          MVG       SVM        IF        AE
MVG  1.000000  0.695238  0.439480  0.963624
SVM  0.695238  1.000000  0.709930  0.667124
IF   0.439480  0.709930  1.000000  0.416603
AE   0.963624  0.667124  0.416603  1.000000

ALL MODELS + ENSEMBLE (Test Holdout)
                   F1  Precision    Recall        F2   AUC_ROC    AUC_PR
MVG          0.516129   1.000000  0.347826  0.400000  0.885701  0.770444
SVM          0.615385   0.750000  0.521739  0.555556  0.850740  0.731559
IF           0.583333   0.560000  0.608696  0.598291  0.809951  0.679511
AE           0.466667   1.000000  0.304348  0.353535  0.862842  0.743406
Hard_Vote    0.516129   1.000000  0.347826  0.400000  0.856791  0.738271
Soft_Mean    0.650000   0.764706  0.565217  0.596330  0.856791  0.738271
Soft_Median  0.619048   0.684211  0.565217  0.585586  0.862394  0.748957

Best ensemble: Soft_Mean (F1=0.650)


## 9. Domain Sub-Scores (USD / Oro / MBS)

**USD sub-score:** mean of signed z-scores of funding stress, DXY, VRP, Treasury flight, USA relative.
**Oro sub-score:** mean of signed z-scores of real yield, DXY (inverted), JPY strength, equity-bond correlation, gold-oil.
**MBS sub-score:** binary rule-based (VIX in [20,28], positive term spread, moderate drawdown, no blocking conditions).


In [10]:
from src.routing import compute_all_subscores
from src.splits import walkforward_split
wf = walkforward_split(embargo_weeks=4, save=False, verbose=False)
subscores_test, subscores_folds = compute_all_subscores(wf['routing'], wf['models'])
print(f"Test sub-scores:\n{subscores_test.describe()}")


Test sub-scores:
          sub_usd     sub_oro    sub_mbs   dxy_chg4w
count  120.000000  120.000000  120.00000  120.000000
mean     0.095804    0.132079    0.02500   -0.001635
std      0.363014    2.132751    0.15678    0.013336
min     -1.123459  -11.824312    0.00000   -0.038678
25%     -0.079300   -0.433718    0.00000   -0.008709
50%      0.112335    0.002199    0.00000   -0.000076
75%      0.317847    0.456666    0.00000    0.005744
max      1.448202   11.918345    1.00000    0.035518


## 10. Routing Engine & Decision Matrix

| Ensemble Signal | Condition | Allocation |
|----------------|-----------|------------|
| Risk-on (0) | — | LEVERED_EQUITY (1.5x) |
| Risk-off (1) | USD high + DXY up | CASH_USD |
| Risk-off (1) | Oro high | GOLD |
| Risk-off (1) | MBS active | MBS |
| Risk-off (1) | Default | CASH_USD |


In [11]:
from src.routing import route_series
from src.backtest import optimize_routing_thresholds, _get_price_series

prices = _get_price_series()

# Optimize thresholds
from src.preprocessing import prepare_X_y
fold_signals = {}
for fold, sc in zip(fold_data['cv_folds'], scalers):
    X_val, y_val = prepare_X_y(fold['val'])
    X_val_s = sc.transform(X_val)
    preds = ensemble.predict_hard(X_val_s)
    fold_signals[fold['fold_id']] = pd.Series(preds, index=X_val.index)

combined = pd.concat(fold_signals.values())
best_thresh, best_calmar = optimize_routing_thresholds(
    combined, subscores_folds, wf['routing'], prices, verbose=True
)

allocations = route_series(pred_series, subscores_test, best_thresh)
print(f"\nAllocation distribution:\n{allocations.value_counts()}")


Optimal thresholds: usd=0.5, oro=0.5
Weighted Calmar: 0.923

Allocation distribution:
LEVERED_EQUITY    103
GOLD                8
CASH_USD            8
MBS                 1
Name: count, dtype: int64


## 11. Backtest Results

Strategy vs benchmarks on the sealed 2019-2021 test holdout.
Transaction costs: LEVERED=5bps, CASH=2bps, GOLD=8bps, MBS=20bps.


In [12]:
from src.backtest import backtest_strategy, benchmark_60_40, benchmark_buyhold, plot_equity_curves

strat_eq, strat_ret, metrics, regime_stats, dd = backtest_strategy(allocations, prices)
b60_eq, b60_ret = benchmark_60_40(prices, strat_eq.index)
bh_eq, bh_ret = benchmark_buyhold(prices, strat_eq.index)

n_years = len(strat_eq) / 52

# Strategy metrics
print("STRATEGY METRICS")
print("=" * 50)
for k, v in metrics.items():
    if 'pct' in k.lower() or k in ['CAGR', 'Annual_Vol', 'Max_DD', 'TC_total']:
        print(f"  {k:20s}: {v:.4f}")
    else:
        print(f"  {k:20s}: {v:.3f}")

print("\nPER-REGIME:")
for regime, stats in regime_stats.items():
    print(f"  {regime}: {stats['n_weeks']}w, total={stats['total_ret']:.2%}")

# Plot
fig = plot_equity_curves(strat_eq, b60_eq, bh_eq, dd, allocations,
                         save_path='outputs/figures/equity_curves.png')
print("\nEquity curve saved to outputs/figures/equity_curves.png")


STRATEGY METRICS
  CAGR                : 0.3522
  Annual_Vol          : 0.1827
  Sharpe              : 1.928
  Sortino             : 2.096
  Max_DD              : -0.1207
  Calmar              : 2.917
  Turnover_pa         : 3.900
  TC_total            : 0.0054
  N_switches          : 9.000

PER-REGIME:
  LEVERED_EQUITY: 103w, total=91.29%
  GOLD: 8w, total=5.01%
  CASH_USD: 8w, total=-0.06%
  MBS: 1w, total=-0.05%



Equity curve saved to outputs/figures/equity_curves.png


## 12. COVID Stress Test

Three windows analyzed:
1. **COVID crash** (2020-02-15 → 2020-04-15): acute market stress
2. **COVID recovery** (2020-04-15 → 2020-12-31): V-shaped rebound
3. **Reflation 2021** (2021-01-01 → end): post-stimulus regime


In [13]:
from src.stress_test import stress_test, plot_stress_timeline

stress_df = stress_test(allocations, prices, verbose=True)

plot_stress_timeline(allocations, prices,
                     save_path='outputs/figures/stress_timeline.png')
print("\nStress timeline saved to outputs/figures/stress_timeline.png")



STRESS TEST RESULTS

COVID_crash (2020-02-15 → 2020-04-15)
  Strategy: -4.57%  (MaxDD -12.07%)
  60/40:    -10.38%  (MaxDD -21.47%)
  BuyHold:  -15.82%  (MaxDD -27.72%)
  Dominant: GOLD, Switches: 3, Lead: 0w

COVID_recovery (2020-04-15 → 2020-12-31)
  Strategy: +25.75%  (MaxDD -9.10%)
  60/40:    +26.38%  (MaxDD -3.74%)
  BuyHold:  +39.60%  (MaxDD -6.05%)
  Dominant: LEVERED_EQUITY, Switches: 6, Lead: 8w

Reflation_2021 (2021-01-01 → 2021-04-20)
  Strategy: +16.90%  (MaxDD -3.48%)
  60/40:    +5.23%  (MaxDD -2.35%)
  BuyHold:  +11.07%  (MaxDD -2.33%)
  Dominant: LEVERED_EQUITY, Switches: 0, Lead: 0w



Stress timeline saved to outputs/figures/stress_timeline.png


## 13. Key Findings & Conclusions

### Model Performance
- **Best single model:** SVM (F1=0.62, AUC-ROC=0.85)
- **Best ensemble:** Soft Mean Voting (F1=0.65) — improves over all single models
- Error correlation ~0.66 confirms ensemble adds value (diverse error patterns)

### Strategy Performance
- The EWS strategy achieves significantly higher risk-adjusted returns than both benchmarks
- Routing engine successfully diversifies risk-off allocations across USD, Gold, and MBS
- COVID crash detected with the ensemble triggering risk-off allocations

### Limitations
1. **Thin validation folds:** Folds 3-5 have very few positives (2, 10, 6), making per-fold metrics noisy
2. **Proxy limitations:** MSCI World proxied by MXUS, Global Aggregate by LUACTRUU
3. **No real-time simulation:** weekly rebalancing with look-ahead on weekly close prices
4. **Credit spreads are log ratios**, not actual OAS spreads
5. **MBS regime is rare** in test period due to strict activation conditions

### Business Implications
- The framework is applicable to institutional portfolio management as a risk overlay
- Domain-specific routing adds intelligence vs. naive cash-only risk-off
- Walk-forward validation with embargo provides realistic out-of-sample assessment
